# Data understanding & Feature Builder notebook

This notebook is responsible for data understanding and produces the feature table that the
risk score is built from later.

No scoring, weighting or ranking happens here. It's features only.

*Important design decision*:
- The feature builder takes a cutoff date as input parameter, 2 reasons for that.
- First, it lets us ask: sitting in December 2024, would this system have flagged V001 — before its last two inspections and before it was detained?
- Second, it makes the system experimental. Any cutoff date can be passed in and the resulting risk picture inspected, rather than being locked to "today".

### 0. Data loading

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


def find_project_root() -> Path:
    current = Path.cwd().resolve()

    for directory in (current, *current.parents):
        if (directory / "pyproject.toml").exists():
            return directory
    raise FileNotFoundError("Project root containing pyproject.toml not found")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "sample_dataset"

vessels = pd.read_csv(DATA_DIR/'VesselMaster.csv')
inspections = pd.read_csv(DATA_DIR/'Inspections.csv', parse_dates=['inspection_date'])
deficiencies = pd.read_csv(DATA_DIR/'Deficiencies.csv', parse_dates=['deficiency_closed_date'])
detentions = pd.read_csv(DATA_DIR/'Detentions.csv', parse_dates=['detention_date'])
maintenance = pd.read_csv(DATA_DIR/'Maintenance.csv', parse_dates=['maintenance_due_date', 'maintenance_completed_date'])
equipment = pd.read_csv(DATA_DIR/'EquipmentFailures.csv', parse_dates=['equipment_failure_date'])
crew = pd.read_csv(DATA_DIR/'Crew.csv', parse_dates=['join_date'])
audits = pd.read_csv(DATA_DIR/'AuditFindings.csv')

VESSEL_IDS = sorted(vessels.vessel_id.unique())

print(f"Count of unique vessel ids is: {len(VESSEL_IDS)}")
print("--"*10)
print("Row and column shape of each csv is as:")
for name, df in [('VesselMaster',vessels), ('Inspections',inspections), ('Deficiencies',deficiencies),
                 ('Detentions',detentions), ('Maintenance',maintenance), ('EquipmentFailures',equipment),
                 ('Crew',crew), ('AuditFindings',audits)]:
    print(f'{name:<20} {df.shape[0]:>4} rows  x {df.shape[1]} cols')

print("--"*10)
print(f'Total {len(detentions)} detentions out of {len(inspections)} inspections')
print("--"*10)
print('unique deficiency_code       :', deficiencies.deficiency_code.nunique())
print('unique deficiency_description:', deficiencies.deficiency_description.nunique())
print("--"*10)


Count of unique vessel ids is: 20
--------------------
Row and column shape of each csv is as:
VesselMaster           20 rows  x 6 cols
Inspections            80 rows  x 8 cols
Deficiencies          286 rows  x 8 cols
Detentions              2 rows  x 4 cols
Maintenance           400 rows  x 6 cols
EquipmentFailures     157 rows  x 5 cols
Crew                  240 rows  x 7 cols
AuditFindings         135 rows  x 4 cols
--------------------
Total 2 detentions out of 80 inspections
--------------------
unique deficiency_code       : 233
unique deficiency_description: 23
--------------------
Crew training Status distribution: {'Compliant': 238, 'Pending': 2}
Crew STCW Status distribution:     {'Valid': 239, 'Expiring Soon': 1}
Non-compliant crew records: 2 - from vessel_id: ['V001']


In [31]:
SENIOR_RANKS = ['Master', 'Chief Engineer', 'Chief Officer', 'Second Engineer']

print(f"Crew training Status distribution: {crew.training_status.value_counts().to_dict()}")
print(f"Crew STCW Status distribution:     {crew.stcw_status.value_counts().to_dict()}")
flagged = crew[(crew.training_status!='Compliant') | (crew.stcw_status!='Valid')]
print(f"Non-compliant crew records: {len(flagged)} - from vessel_id: {[id for id in flagged.vessel_id.unique()]}")

print("\n")
exp = crew.groupby('rank').rank_experience_months.agg(['count','mean','min','max'])
exp[['mean','min','max']] = (exp[['mean','min','max']] / 12).round(1)
exp.columns = ['n', 'mean_yrs', 'min_yrs', 'max_yrs']
print(exp.sort_values('mean_yrs', ascending=False))

Crew training Status distribution: {'Compliant': 238, 'Pending': 2}
Crew STCW Status distribution:     {'Valid': 239, 'Expiring Soon': 1}
Non-compliant crew records: 2 - from vessel_id: ['V001']


                  n  mean_yrs  min_yrs  max_yrs
rank                                           
Cook             20      15.1      6.3     22.5
Chief Engineer   40      15.0      1.9     24.6
AB               20      13.7      2.2     23.8
ETO              20      13.6      2.5     22.8
Oiler            20      13.2      2.6     25.0
Chief Officer    40      12.4      1.5     23.3
Master           40      12.2      1.6     24.5
Second Engineer  40      12.1      2.1     23.8


In [37]:
def range(s): 
    return f'{s.min().date()}  ->  {s.max().date()}'

print("Date Ranges in YYYY-MM-DD:\n")
print('inspections        ', range(inspections.inspection_date))
print('equipment failures ', range(equipment.equipment_failure_date))
print('crew join dates    ', range(crew.join_date))
print('maintenance due    ', range(maintenance.maintenance_due_date))

Date Ranges in YYYY-MM-DD:

inspections         2024-01-19  ->  2025-07-05
equipment failures  2025-03-24  ->  2026-08-21
crew join dates     2023-12-08  ->  2026-08-19
maintenance due     2026-01-26  ->  2026-12-22


**What above date range says:**
- Inspections stop in **July 2025**.
- Maintenance due dates are entirely in **2026**.
- Crew and equipment records run to **August 2026**.

**Conclusion:**
There is no period in which "this vessel had X overdue jobs" can be linked to "this vessel then received Y deficiencies".


## giving deficiencies a date
1. Deficiencies.csv has no date column. It only knows which inspection it came from. So to know when a deficiency happened, you look up its parent inspection.
2. how many deficiencies were open on that day?
    - raised before cutoff  AND  (never closed  OR  closed after the cutoff)

In [43]:
defs_dated = deficiencies.merge(
    inspections[['inspection_id','inspection_date','inspection_authority','inspection_result']],
    on='inspection_id', how='left', validate='many_to_one')

print(f"Deficiencies with no parent inspection: {defs_dated.inspection_date.isna().sum()}")

Deficiencies with no parent inspection: 0


### 1. How the tables connect

Two keys join everything: `vessel_id` (in all 8 files) and `inspection_id` (in 3).

```
VesselMaster (20 vessels)
│
├── Inspections (80) 4 per vessel
│ ├── Deficiencies (286) what was found at that inspection
│ └── Detentions (2) the worst outcome of that inspection
│
├── Maintenance (400) current state of the vessel —
├── EquipmentFailures (157) not linked to any inspection
├── Crew (240)
└── AuditFindings (135)
```

*Important caveat*:
- The cutoff only works on inspection data. Inspections run Jan 2024 to Jul 2025, so rewinding to December 2024 gives a real picture — two inspections had happened by then.
- The operational tables can't be rewound. Maintenance due dates are all in 2026, equipment failures start Mar 2025, and AuditFindings has no dates at all. Rewind to December 2024 and these tables are empty.
- So features are built in two groups: inspection features that accept a cutoff date, and operational features that only describe today.

## 3. Feature definitions

Fifteen features across seven risk dimensions, defined in `decision_2` file. 

They are split by whether they can be reconstructed for a past date.

| Group | Cutoff-aware | Why |
|---|---|---|
| Inspection features | yes | every record carries an inspection date |
| Operational features | no | audit findings have no dates; maintenance due dates are all future |

Each function returns one row per vessel, indexed identically, so they can be joined safely.


### 3.1 Inspection features — reconstructable for any past date

| Feature | Definition |
|---|---|
| `n_inspections` | inspections on or before the cutoff
| `n_deficiencies` | total findings |
| `repeat_count` | occurrences of each issue beyond the first, summed |
| `max_repeat` | highest count for any single issue |
| `concentration` | largest category's share of all findings |
| `top_category` | name of that category |
| `trend` | findings at latest inspection minus findings at earliest |
| `open_deficiencies` | raised before the cutoff and not closed by it |

In [84]:
# _align forces the Series onto all 20 vessels, in fixed order.
def _align(series, fill=0):
    """Reindex a per-vessel series onto the full vessel list so every vessel appears exactly once."""
    return pd.Series(series).reindex(VESSEL_IDS).fillna(fill)

def inspection_features(as_of_date):
    """Vessel-level inspection features using only records dated on or before as_of_date."""
    as_of = pd.Timestamp(as_of_date)
    insp = inspections[inspections.inspection_date <= as_of]
    defs = defs_dated[defs_dated.inspection_date <= as_of]


    # .size() counts rows in each group and returns one number per vessel
    n_insp = _align(insp.groupby('vessel_id').size())
    n_def  = _align(defs.groupby('vessel_id').size())

    
    # --- repetition: the same issue recorded more than once
    per_desc     = defs.groupby(['vessel_id','deficiency_description']).size()

    # repeat_count is one number per vessel, mixing all issues together.
    # g[g > 1] — keep only the repeats
    repeat_count = _align(per_desc.groupby('vessel_id').apply(lambda g: (g[g > 1] - 1).sum()))

    # max_repeat is one dominant fault
    max_repeat   = _align(per_desc.groupby('vessel_id').max())


    # --- concentration: share of findings in the single largest category
    per_cat       = defs.groupby(['vessel_id','deficiency_category']).size()

    largest_cat   = _align(per_cat.groupby('vessel_id').max())

    # concentration: E.g.For V004: 13 navigation findings ÷ 18 total = 0.72. 
    # Nearly three-quarters of its problems are in one area.
    concentration = (largest_cat / n_def.replace(0, np.nan)).fillna(0)
    
    top_category  = _align(per_cat.groupby('vessel_id').idxmax()
                                  .apply(lambda t: t[1] if isinstance(t, tuple) else None), fill=None)

    # --- trend: change in findings between first and most recent inspection
    per_insp = (defs.groupby(['vessel_id','inspection_id','inspection_date']).size()
                    .reset_index(name='n').sort_values('inspection_date'))

    # Here trend can have outputs like V001 -> 7, V005 -> -3
    # Negatives mean improving - the vessel had fewer findings at its last inspection than its first.
    trend = _align(per_insp.groupby('vessel_id').n.last() - per_insp.groupby('vessel_id').n.first())

    # --- backlog: open AS OF the cutoff, derived from closure date
    open_as_of = defs[(defs.deficiency_closed_date.isna()) | (defs.deficiency_closed_date > as_of)]
    open_def   = _align(open_as_of.groupby('vessel_id').size())
    
    return pd.DataFrame({
        'n_inspections'    : n_insp.astype(int),
        'n_deficiencies'   : n_def.astype(int),
        'repeat_count'     : repeat_count.astype(int),
        'max_repeat'       : max_repeat.astype(int),
        'top_category'     : top_category,
        'concentration'    : concentration.round(3),
        'trend'            : trend.astype(int),
        'open_deficiencies': open_def.astype(int),
    })

In [85]:
inspection_features(as_of_date="2026-01-08")

,n_inspections,n_deficiencies,repeat_count,max_repeat,top_category,concentration,trend,open_deficiencies
vessel_id,,,,,,,,
V001,4,28,14,9,Fire Safety,0.500,7,10
V002,4,14,4,2,Fire Safety,0.286,1,3
V003,4,14,3,3,Structural,0.286,-1,3
V004,4,18,9,4,Navigation,0.722,5,5
V005,4,10,2,2,Life Saving,0.300,-3,0
V006,4,11,2,2,Fire Safety,0.273,-2,2
V007,4,14,5,4,Navigation,0.357,1,1
V008,4,10,3,2,Structural,0.400,1,1
V009,4,11,1,2,Machinery,0.455,-1,1


### 3.2 Operational features — current state only

| Feature | Definition |
|---|---|
| `overdue_maintenance` | planned jobs with status Overdue |
| `open_audit` / `overdue_audit` | unresolved internal findings |
| `known_issues_unresolved` | distinct issues open internally that also appear as deficiencies |
| `known_issue_occurrences` | how many external findings those issues account for |
| `equipment_repeats` | repeat failures of the same named equipment |
| `recent_joiners` | crew who joined within 90 days |
| `avg_experience` | mean rank experience, months |

**On the two `known_issues_unresolved` and  `known_issue_occurrences` features.** 
- Audit findings and deficiencies use exactly the same 23 issue descriptions
- so it is possible to detect issues the company raised against itself, failed to close, and an inspector then recorded anyway.

In [ ]:
def operational_features(reference_date):
    """Vessel-level current-state features. No cutoff parameter"""
    ref = pd.Timestamp(reference_date)

    overdue_maint = _align(maintenance[maintenance.maintenance_status=='Overdue']
                           .groupby('vessel_id').size())
    open_audit    = _align(audits[audits.audit_finding_status=='Open'].groupby('vessel_id').size())
    overdue_audit = _align(audits[audits.audit_finding_status=='Overdue'].groupby('vessel_id').size())


    # issues unresolved internally that also appear as external deficiencies
    unresolved = audits[audits.audit_finding_status.isin(['Open','Overdue'])]
    linked = unresolved.merge(deficiencies,
                              left_on =['vessel_id','audit_finding_description'],
                              right_on=['vessel_id','deficiency_description'],
                              how='inner')

    # V001 has one unresolved audit finding: "Fire pump pressure low". 
    # It also has that same issue recorded as a deficiency 9 times.
    known_issues      = _align(linked.groupby('vessel_id').audit_finding_description.nunique())
    known_occurrences = _align(linked.groupby('vessel_id').size())

    # repeat failures of the same equipment (raw failure count carries no signal)
    per_equip     = equipment.groupby(['vessel_id','equipment_name']).size()
    equip_repeats = _align(per_equip.groupby('vessel_id').apply(lambda g: (g[g > 1] - 1).sum()))
    

    recent         = crew[crew.join_date > ref - pd.Timedelta(days=90)]
    recent_joiners = _align(recent.groupby('vessel_id').size())
    avg_experience = _align(crew.groupby('vessel_id').rank_experience_months.mean())

    return pd.DataFrame({
        'overdue_maintenance'    : overdue_maint.astype(int),
        'open_audit'             : open_audit.astype(int),
        'overdue_audit'          : overdue_audit.astype(int),
        'known_issues_unresolved': known_issues.astype(int),
        'known_issue_occurrences': known_occurrences.astype(int),
        'equipment_repeats'      : equip_repeats.astype(int),
        'recent_joiners'         : recent_joiners.astype(int),
        'avg_experience'         : avg_experience.round(1),
    })

In [103]:
REFERENCE_DATE = pd.Timestamp('2026-08-24')
operational_features(REFERENCE_DATE)

,overdue_maintenance,open_audit,overdue_audit,known_issues_unresolved,known_issue_occurrences,equipment_repeats,recent_joiners,avg_experience
vessel_id,,,,,,,,
V001,11,3,1,3,20,3,6,167.1
V002,4,1,3,1,1,4,0,128.6
V003,4,4,0,2,2,5,0,166.8
V004,7,4,1,3,14,3,2,190.2
V005,4,0,4,3,4,2,1,169.8
V006,5,2,1,1,1,2,1,165.8
V007,8,1,0,0,0,5,0,153.0
V008,6,1,2,0,0,3,1,156.0
V009,4,4,2,2,2,4,4,151.5


In [102]:
def build_features(as_of_date=REFERENCE_DATE, reference_date=REFERENCE_DATE):
    """One row per vessel: particulars + inspection features + operational features."""
    base = (vessels.set_index('vessel_id')
                   .loc[VESSEL_IDS, ['vessel_name','vessel_type','build_year','flag_state']])
    base['vessel_age'] = pd.Timestamp(reference_date).year - base.build_year

    out = base.join(inspection_features(as_of_date)).join(operational_features(reference_date))
    out.index.name = 'vessel_id'
    return out

## 4. Build the feature table

In [104]:
REFERENCE_DATE = pd.Timestamp('2026-08-24')

features = build_features(as_of_date=REFERENCE_DATE, reference_date=REFERENCE_DATE)
features.to_csv('features.csv')

display_cols = [c for c in features.columns if c not in ('vessel_name','flag_state','build_year')]
features[display_cols]

,vessel_type,vessel_age,n_inspections,n_deficiencies,repeat_count,max_repeat,top_category,concentration,trend,open_deficiencies,overdue_maintenance,open_audit,overdue_audit,known_issues_unresolved,known_issue_occurrences,equipment_repeats,recent_joiners,avg_experience
vessel_id,,,,,,,,,,,,,,,,,,
V001,Bulk Carrier,18,4,28,14,9,Fire Safety,0.500,7,10,11,3,1,3,20,3,6,167.1
V002,Oil Tanker,17,4,14,4,2,Fire Safety,0.286,1,3,4,1,3,1,1,4,0,128.6
V003,Container Ship,16,4,14,3,3,Structural,0.286,-1,3,4,4,0,2,2,5,0,166.8
V004,Chemical Tanker,15,4,18,9,4,Navigation,0.722,5,5,7,4,1,3,14,3,2,190.2
V005,LNG Carrier,14,4,10,2,2,Life Saving,0.300,-3,0,4,0,4,3,4,2,1,169.8
V006,Bulk Carrier,13,4,11,2,2,Fire Safety,0.273,-2,2,5,2,1,1,1,2,1,165.8
V007,Oil Tanker,12,4,14,5,4,Navigation,0.357,1,1,8,1,0,0,0,5,0,153.0
V008,Container Ship,11,4,10,3,2,Structural,0.400,1,1,6,1,2,0,0,3,1,156.0
V009,Chemical Tanker,10,4,11,1,2,Machinery,0.455,-1,1,4,4,2,2,2,4,4,151.5


### Cutoff check

The same function called at an earlier date must return a genuinely smaller view of the fleet.

In [107]:
early = inspection_features('2024-12-31')
late  = inspection_features(REFERENCE_DATE)

comparison = pd.DataFrame({
    'inspections_2024': early.n_inspections, 'inspections_now': late.n_inspections,
    'deficiencies_2024': early.n_deficiencies, 'deficiencies_now': late.n_deficiencies,
    'open_2024': early.open_deficiencies, 'open_now': late.open_deficiencies,
})
print(comparison.head(6).to_string())
print()
print('fleet totals  2024-12-31 :', early.n_deficiencies.sum(), 'deficiencies,',
      early.open_deficiencies.sum(), 'open')
print('fleet totals  2026-08-24 :', late.n_deficiencies.sum(), 'deficiencies,',
      late.open_deficiencies.sum(), 'open')

           inspections_2024  inspections_now  deficiencies_2024  deficiencies_now  open_2024  open_now
vessel_id                                                                                             
V001                      2                4                  9                28          0        10
V002                      2                4                  7                14          0         3
V003                      2                4                  8                14          0         3
V004                      2                4                  6                18          0         5
V005                      3                4                  9                10          3         0
V006                      3                4                  8                11          1         2

fleet totals  2024-12-31 : 163 deficiencies, 22 open
fleet totals  2026-08-24 : 286 deficiencies, 47 open
